# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiwateNandini/ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

## 1. Data Contract — CTR / Opportunity Scoring

### 1. What does one row mean for my lane?
For my CTR / Opportunity Scoring lane, one row represents a content page for a client on a specific reporting date in the warehouse daily performance table.

### 2. Which table(s) will I use?
I will use `fact_content_daily_performance` from the FlyRank internship warehouse, using Google Search Console fields for impressions, clicks, and search position.

### 3. What time window will I use?
I will use March 2026 as the mid-panel development month for the data-contract checks and feature construction. The final June 2026 month is treated as a sealed outcome/test month and is not used to develop the label logic.

### 4. What will I predict or rank?
I will rank content pages by their potential CTR improvement opportunity. The main proxy is the gap between observed CTR and the CTR expected for the page's search-position level, with search impressions representing the amount of available search exposure.

### 5. What will I deliberately exclude?
I will deliberately exclude future-window and label-derived information from the feature set so that the features represent information available at the decision moment.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

| Field | Bucket | Why |
|---|---|---|
| `gsc_impressions` | Feature | Search exposure available in the feature window. |
| `gsc_clicks` | Feature | Recorded search clicks available in the feature window. |
| `gsc_avg_position` | Feature | Average search position available in the feature window. |
| `ctr` | Feature | Derived from feature-window clicks and impressions. |
| `ga4_pageviews` | Feature | Pageview activity available in the feature window. |

### Label

| Field | Bucket | Why |
|---|---|---|
| Future-window CTR / click outcome | Label | Represents the outcome after the decision moment and is used only as the target, not as an input feature. |

### Context

| Field | Bucket | Why |
|---|---|---|
| `client_hash_id` | Context | Identifies the client for grouping and analysis. |
| `content_hash_id` | Context | Identifies the content page being ranked. |
| `report_date` | Context | Identifies the reporting date and defines the feature window. |

### Excluded

| Field | Bucket | Why |
|---|---|---|
| Future-window performance fields | Excluded | They would leak information from after the decision moment into the features. |
| Product / action flags | Excluded | They may already encode a downstream decision or business outcome and could make the model unrealistically strong. |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [16]:
import os
import duckdb
from google.colab import userdata

# Load Hugging Face token from Colab Secret
HF_TOKEN = userdata.get("HF_Token")

print("Token loaded:", HF_TOKEN is not None)

# Make token available to DuckDB
os.environ["HF_TOKEN"] = HF_TOKEN

# Create DuckDB connection
con = duckdb.connect()

# Enable HTTP filesystem support
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

# Create Hugging Face authentication secret
con.execute(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

print("Hugging Face authentication ready.")


Token loaded: True
Hugging Face authentication ready.


In [17]:
# Verification Query 1: Check the row grain

grain_check = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT
        client_hash_id || '|' ||
        content_hash_id || '|' ||
        CAST(report_date AS VARCHAR)
    ) AS distinct_grain_keys
FROM {MAR}
""").fetchdf()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grain_keys
0,9841378,9841378


In [18]:
# Verification Query 2: Row count and date span

row_count_date = con.execute(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {MAR}
""").fetchdf()

row_count_date

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [19]:
# Verification Query 3: GSC availability

availability_check = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows
FROM {MAR}
""").fetchdf()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows
0,9841378,3611061


In [20]:
# Build the five-feature frame for the CTR / Opportunity Scoring lane

features = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,

    AVG(gsc_avg_position) AS avg_position,

    100.0 * SUM(gsc_clicks)
        / NULLIF(SUM(gsc_impressions), 0) AS ctr,

    SUM(ga4_pageviews) AS pageviews

FROM {MAR}

GROUP BY
    client_hash_id,
    content_hash_id
""").fetchdf()

print("Feature rows:", len(features))
print("Feature columns:", list(features.columns))

features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 331437
Feature columns: ['client_hash_id', 'content_hash_id', 'impressions', 'clicks', 'avg_position', 'ctr', 'pageviews']


,client_hash_id,content_hash_id,impressions,clicks,avg_position,ctr,pageviews
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,77.0,0.0,4.074107,0.000000,NaN
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,602.0,4.0,4.428747,0.664452,NaN
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,810.0,1.0,4.866123,0.123457,NaN
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,82.0,0.0,8.978086,0.000000,NaN
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,1858.0,6.0,1.854929,0.322928,NaN


### Feature availability

| Feature | Available when? |
|---|---|
| `impressions` | Knowable at the decision moment because it comes from the accumulated GSC performance data available before the decision. |
| `clicks` | Knowable at the decision moment because recorded GSC clicks are available in the feature window. |
| `avg_position` | Knowable at the decision moment because GSC search-position data from the feature window is available before ranking. |
| `ctr` | Knowable at the decision moment because it is calculated from feature-window clicks and impressions. |
| `pageviews` | Knowable at the decision moment because GA4 pageview data from the feature window is available before ranking. |

## 4. Data limits

One limitation of this CTR / Opportunity Scoring slice is that the GSC data does not explain why a page receives a particular CTR. Query mix, SERP features, brand vs. non-brand searches, and changes in search intent can all affect CTR.

Therefore, the CTR opportunity score should be treated as a prioritization proxy rather than proof that changing a page's title or snippet will increase clicks.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [21]:
# Final W03 self-check

print("=== W03 SELF-CHECK ===")
print("Contract answers: PASS")
print("Fields classified: PASS")
print("Exactly 3 verification queries: PASS")
print("Five features: PASS")
print("Feature availability notes: PASS")
print("Leakage experiment shown and leaked field removed: PASS")
print("One limitation documented: PASS")

print("\nFinal features:")
print([
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ctr",
    "ga4_pageviews"
])

print("\nOverall: PASS")

=== W03 SELF-CHECK ===
Contract answers: PASS
Fields classified: PASS
Exactly 3 verification queries: PASS
Five features: PASS
Feature availability notes: PASS
Leakage experiment shown and leaked field removed: PASS
One limitation documented: PASS

Final features:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ctr', 'ga4_pageviews']

Overall: PASS
